# Commercial Fleet Route Optimization - Step 2: Classical Preprocessing
This notebook parses the SUMO network file (`osm.net.xml.gz`), builds a NetworkX directed graph restricted to roads a delivery van can use (largest strongly connected component), and computes candidate paths with Yen's $k$-shortest simple paths algorithm.

**Outputs for the QUBO step:** `paths[d]`, `C[(d,p)]` (free-flow seconds), `E[(d,p)]` (edge-ID sets), `S[(a,b)]` (shared-edge counts between routes).

In [ ]:
# Install required libraries if not already installed
%pip install sumolib networkx matplotlib

In [ ]:
import os
import random
from itertools import islice, combinations

import sumolib
import networkx as nx
import matplotlib.pyplot as plt

## 1. Build the road graph
Only vehicle-accessible edges are kept (`allows("delivery")`), zero-speed edges are skipped, parallel edges keep the faster one, and the graph is reduced to its largest strongly connected component so every depot→destination path exists.

In [ ]:
net_path = os.path.join(os.curdir, "SUMO_GEN_SCENARIO", "osm.net.xml.gz")
assert os.path.exists(net_path), f"File not found: {net_path}"
print(f"Found network file at: {net_path}")

net = sumolib.net.readNet(net_path)

G = nx.DiGraph()
for edge in net.getEdges():
    # Skip internal junction connectors and roads a delivery van cannot drive on
    if edge.isSpecial() or not edge.allows("delivery"):
        continue
    speed = edge.getSpeed()
    if speed <= 0:
        continue

    u = edge.getFromNode().getID()
    v = edge.getToNode().getID()
    tt = edge.getLength() / speed          # free-flow travel time (s)

    # DiGraph cannot hold parallel edges: keep the faster one
    if G.has_edge(u, v) and G[u][v]["weight"] <= tt:
        continue
    G.add_edge(u, v, edge_id=edge.getID(), length=edge.getLength(), weight=tt)

print(f"Before SCC reduction: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Keep only the largest strongly connected component (guarantees reachability)
core = max(nx.strongly_connected_components(G), key=len)
G = G.subgraph(core).copy()

for n in G.nodes:
    G.nodes[n]["pos"] = net.getNode(n).getCoord()

print(f"Largest SCC:          {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 2. Scenario configuration
Depot and destinations are sampled from the whole strongly connected component (not from file order) to avoid spatially clustered stops.

In [ ]:
NUM_VANS = 3
NUM_DESTINATIONS = 5

random.seed(42)  # reproducible selection
nodes = sorted(G.nodes)
DEPOT_NODE = random.choice(nodes)
DESTINATION_NODES = random.sample([n for n in nodes if n != DEPOT_NODE], NUM_DESTINATIONS)

print(f"Configured Fleet Size: {NUM_VANS} vans")
print(f"Depot Node: {DEPOT_NODE}")
print(f"Target Destinations ({len(DESTINATION_NODES)} total): {DESTINATION_NODES}")

## 3. Candidate routes, costs and edge overlap
Route cost $C_{d,p}$ and edge sets depend only on $(d,p)$, so they are computed once (not per van). The first `SKIP_HOPS` edges out of the depot are excluded from the overlap sets, since every route shares them and they would swamp the congestion signal.

In [ ]:
def get_candidate_routes(graph, source, target, k=1):
    """Up to k loopless shortest paths (Yen's algorithm) as node lists."""
    try:
        return list(islice(nx.shortest_simple_paths(graph, source, target, weight="weight"), k))
    except nx.NetworkXNoPath:
        return []

K_PATHS = 1      # use 2 for a non-trivial route-choice (cost) term
SKIP_HOPS = 2    # ignore the first edges out of the depot in the overlap count

paths = {}   # d -> [p, ...]
C = {}       # (d, p) -> free-flow travel time (s)
E = {}       # (d, p) -> set of edge IDs (depot-adjacent edges removed)
NODE_PATHS = {}

for d in DESTINATION_NODES:
    node_paths = get_candidate_routes(G, DEPOT_NODE, d, k=K_PATHS)
    assert node_paths, f"No path to {d}"
    paths[d] = list(range(len(node_paths)))
    for p, route in enumerate(node_paths):
        legs = list(zip(route, route[1:]))
        NODE_PATHS[d, p] = route
        C[d, p] = sum(G[u][v]["weight"] for u, v in legs)
        E[d, p] = {G[u][v]["edge_id"] for u, v in legs[SKIP_HOPS:]}

routes = list(C)
S = {(a, b): len(E[a] & E[b]) for a, b in combinations(routes, 2)}
vans = list(range(1, NUM_VANS + 1))

print(f"Routes: {len(routes)}  |  QUBO variables: {len(vans) * len(routes)}")

## 4. Diagnostics (check before choosing penalty weights)

In [ ]:
print("Route costs (s):")
for a in routes:
    print(f"  {a}: {C[a]:8.1f} s, {len(NODE_PATHS[a]) - 1} edges, {len(E[a])} counted for overlap")

s_vals = list(S.values())
print(f"\nShared-edge counts S: min={min(s_vals)}, max={max(s_vals)}, "
      f"mean={sum(s_vals)/len(s_vals):.1f}, zeros={sum(v == 0 for v in s_vals)}/{len(s_vals)}")
if all(v == 0 for v in s_vals):
    print("WARNING: no overlap, so there is no congestion signal to optimise.")

# Normalise costs so lam_cong and the cost term are comparable
C_MAX = max(C.values())
C_norm = {a: c / C_MAX for a, c in C.items()}
print(f"\nMax cost (normaliser): {C_MAX:.1f} s")

In [ ]:
# Optional: visualise depot, destinations and routes
fig, ax = plt.subplots(figsize=(8, 8))
for u, v in G.edges:
    (x1, y1), (x2, y2) = G.nodes[u]["pos"], G.nodes[v]["pos"]
    ax.plot([x1, x2], [y1, y2], color="lightgray", lw=0.3, zorder=1)
for a, route in NODE_PATHS.items():
    xs, ys = zip(*(G.nodes[n]["pos"] for n in route))
    ax.plot(xs, ys, lw=1.5, zorder=2, label=f"dest {a[0]} p{a[1]}")
ax.scatter(*G.nodes[DEPOT_NODE]["pos"], c="black", marker="s", s=80, zorder=3, label="depot")
for d in DESTINATION_NODES:
    ax.scatter(*G.nodes[d]["pos"], marker="*", s=120, zorder=3)
ax.set_aspect("equal"); ax.set_title("Depot, destinations and candidate routes")
ax.legend(fontsize=6, loc="best")
plt.show()

## Next step
`paths`, `C_norm`, `S`, `vans` feed directly into the QUBO builder (`build_qubo`) with e.g. `Qv={1: 2, 2: 2, 3: 1}`.

**Thesis notes:** network filtered to vehicle-accessible edges and the largest strongly connected component; routes from Yen's $k$-shortest paths (no $A^*$ heuristic is used); the first `SKIP_HOPS` depot edges are excluded from the overlap count.